[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Imbalanced Classes


## What you will be able to do

Judge a model of a rare event against a baseline that never predicts it, report recall, precision and
average precision in place of accuracy, and keep the rare class in every split.

## The idea

### The problem

Frost in June kills seedlings, and a farming cooperative wants to warn the growers near the stations
that get one. In the network, 59 of 1,200 stations had a June frost, one in twenty. A model trained to
predict it reports an accuracy of 97 percent on stations it has not seen, which sounds like a model
ready to use.

A model that never predicts frost at all scores 95 percent on the same stations. It has learned
nothing and warns nobody, and it is wrong only at the stations that matter. Against that baseline,
97 percent is two points of improvement, and the number that would tell the cooperative what it needs,
how many frosts the model catches, is not in it.

A rare class causes smaller trouble too. A quarter of 1,200 stations held back for testing contains
about 15 frost stations, but a split that ignores the class can put 10 in the test set or 21, and
scores on so few cases move a long way between one split and the next.

### What an imbalanced class is

> A classification problem has an **imbalanced class** when one class is much rarer than the others.
> A **baseline**, such as scikit-learn's `DummyClassifier`, predicts without looking at the features,
> and any model worth using beats it on a measure that counts the rare class. **Balanced accuracy** is
> the mean of the recall of each class, and **average precision** summarizes precision across every
> threshold from the model's probabilities.

### Why it works that way

Accuracy counts every station equally, so with 95 percent of them clear, getting the clear ones right
is almost the whole score. A model can raise its accuracy by predicting frost less often, because each
frost it gives up costs one station and each false alarm it avoids saves one, and there are far more
chances for false alarms. That is the opposite of what the cooperative wants.

Measures built around the rare class do not have this blind spot. Recall asks how many frosts were
found, and a baseline that never predicts frost scores 0 on it. Balanced accuracy averages the recall
of frost and of no frost, so the baseline scores one half however rare frost is. Average precision
adds up the precision at each threshold, weighted by how much recall rises there, which is close to
the area under the curve of precision against recall, and a baseline scores the share of the rare
class, here 0.05.

A model can also be told that the rare class matters more. `class_weight="balanced"` gives each frost
station a larger weight, so that the frost stations together count for as much as all the clear ones,
and the model gives up some accuracy to find more frosts. Stratified splits, with `stratify` and
`StratifiedKFold`, keep the share of frost the same in every part, so that no part is left with too
few frost stations to score. The scores still move from one part to the next, because each part
holds only a dozen or so frosts.

### Where this shows up

Fraud, equipment failure, rare diseases and a click on an ad are all rare, and all are what a
model is wanted for. The **Classification Metrics** notebook defined precision, recall and the
threshold, and the **Cross-Validation** notebook introduced stratified folds.

### What this notebook covers

- how rare June frost is, and a split that keeps its share
- `DummyClassifier`, the baseline any model has to beat
- a logistic regression, its accuracy, and the frosts it misses
- balanced accuracy, and `class_weight="balanced"`
- average precision from probabilities, and stratified cross-validation
- four errors, among them a precision with nothing to divide by

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

frost = [False] * 19 + [True]
features = [[n] for n in range(20)]

never = DummyClassifier(strategy="most_frequent").fit(features, frost)
print(never.predict(features[-3:]))
print(accuracy_score(frost, never.predict(features)), recall_score(frost, never.predict(features)))
```

```
[False False False]
0.95 0.0
```

The baseline predicts the most common class, `False`, for every row. That is right 19 times out of
20, an accuracy of 0.95, and it finds none of the one frost, a recall of 0.0. Accuracy would let it
pass, and recall shows it for what it is.

## Setup

Two libraries for the data, scikit-learn, and a baseline, a model and the metrics that judge them.

- `numpy` and `pandas` build the stations and the tables of results, and the cell prints
  scikit-learn's version
- `train_test_split`, `StratifiedKFold` and `cross_val_score` split the stations and score the models
- `DummyClassifier` is the baseline, and `make_pipeline`, `StandardScaler` and `LogisticRegression` the
  model
- `accuracy_score`, `balanced_accuracy_score`, `recall_score`, `precision_score`, `confusion_matrix` and
  `average_precision_score` are the metrics
- `warnings` catches the warning that Common errors prints

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. The column
this notebook predicts is `june_frost`, `True` for a station with a frost in June, and `FEATURES`
names the five numeric columns it predicts from.

In [1]:
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### How rare frost is

In [2]:
X = network[FEATURES]
y = network["june_frost"]

print(int(y.sum()), "of", len(y), "stations had a frost in June, a share of", round(y.mean(), 3))

59 of 1200 stations had a frost in June, a share of 0.049


### A split that keeps the share

`stratify=y` puts the same share of frost stations in the training and test sets:

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

print("training:", int(y_train.sum()), "frost stations of", len(y_train))
print("test:    ", int(y_test.sum()), "frost stations of", len(y_test))

training: 44 frost stations of 900
test:     15 frost stations of 300


The test set has 15 frost stations among 300, and every score below rests on those 15.

### The baseline

`DummyClassifier(strategy="most_frequent")` learns only which class is most common, and predicts it
for every station:

In [4]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_says = baseline.predict(X_test)

print("predicts frost at", int(baseline_says.sum()), "stations")
print("accuracy:", round(accuracy_score(y_test, baseline_says), 3))
print("recall:  ", recall_score(y_test, baseline_says))

predicts frost at 0 stations
accuracy: 0.95
recall:   0.0


It predicts frost nowhere, and scores 0.95, the share of clear stations in the test set. Any model has
to be judged against that number, and against a recall of 0.

### A model, and the frosts it misses

In [5]:
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
says = model.predict(X_test)

print("accuracy: ", round(accuracy_score(y_test, says), 3))
print("precision:", round(precision_score(y_test, says), 3))
print("recall:   ", round(recall_score(y_test, says), 3))
pd.DataFrame(confusion_matrix(y_test, says), index=["clear", "frost"], columns=["predicted clear", "predicted frost"])

accuracy:  0.973
precision: 0.889
recall:    0.533


,predicted clear,predicted frost
clear,284,1
frost,7,8


An accuracy of 0.973 is 0.023 above the baseline's. The confusion matrix shows what that means. The
model found 8 of the 15 frosts and raised one false alarm, so its warnings are nearly always right, a
precision of 0.889, and it misses almost half the frosts, a recall of 0.533.

### Balanced accuracy

`balanced_accuracy_score` averages the recall of each class, so the baseline cannot hide behind the
common one:

In [6]:
print("baseline:", round(balanced_accuracy_score(y_test, baseline_says), 3))
print("model:   ", round(balanced_accuracy_score(y_test, says), 3))

baseline: 0.5
model:    0.765


The baseline scores 0.5, the score of a model that tells the classes apart no better than chance,
and the model 0.765. The gap is far wider than accuracy's 0.95 against 0.973.

### Weighting the rare class

`class_weight="balanced"` weights each class by the inverse of its share, so the 44 frost stations in
training count for as much as the 856 clear ones:

In [7]:
weighted = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(X_train, y_train)
weighted_says = weighted.predict(X_test)

print("accuracy:         ", round(accuracy_score(y_test, weighted_says), 3))
print("balanced accuracy:", round(balanced_accuracy_score(y_test, weighted_says), 3))
print("precision:        ", round(precision_score(y_test, weighted_says), 3))
print("recall:           ", round(recall_score(y_test, weighted_says), 3))
pd.DataFrame(confusion_matrix(y_test, weighted_says), index=["clear", "frost"], columns=["predicted clear", "predicted frost"])

accuracy:          0.897
balanced accuracy: 0.882
precision:         0.31
recall:            0.867


,predicted clear,predicted frost
clear,256,29
frost,2,13


The weighted model found 13 of the 15 frosts, a recall of 0.867, and paid for them with 29 false
alarms, a precision of 0.31. Its accuracy fell to 0.897, below the baseline's 0.95, while its balanced
accuracy rose to 0.882. For a cooperative that would rather warn too often than lose a crop, this
model is the better one, and accuracy ranks it last.

### Average precision from probabilities

`average_precision_score` takes probabilities and summarizes precision at every threshold, so it
judges the model before any threshold is chosen:

In [8]:
for name, fitted in [("baseline", baseline), ("model", model), ("weighted", weighted)]:
    chance = fitted.predict_proba(X_test)[:, 1]
    print(f"{name:8} {average_precision_score(y_test, chance):.3f}")

top_model = set(np.argsort(-model.predict_proba(X_test)[:, 1], kind="stable")[:15])
top_weighted = set(np.argsort(-weighted.predict_proba(X_test)[:, 1], kind="stable")[:15])
print("same 15 stations ranked highest:", top_model == top_weighted)

baseline 0.050
model    0.720
weighted 0.728
same 15 stations ranked highest: True


The baseline scores 0.05, the share of frost in the test set, which is what guessing earns on this
measure. The two models score 0.72 and 0.728. Weighting changed which stations cross the threshold of
one half, and both models put the same 15 stations at the top of their ranking, so their average
precision is close while their recall differs. The **Classification Metrics** notebook shows how to
choose a threshold from the costs.

### Stratified cross-validation

`StratifiedKFold` keeps the share of frost the same in every fold, and `scoring="recall"` scores each
fold on the rare class:

In [9]:
folds = StratifiedKFold(5, shuffle=True, random_state=0)
print("frost stations in each fold:", [int(y.iloc[test].sum()) for _, test in folds.split(X, y)])

recalls = cross_val_score(weighted, X, y, cv=folds, scoring="recall")
print("recall in each fold:", recalls.round(3), "| mean:", round(recalls.mean(), 3))

frost stations in each fold: [11, 12, 12, 12, 12]
recall in each fold: [0.909 0.917 1.    0.667 0.917] | mean: 0.882


Each fold holds 11 or 12 of the 59 frost stations, and recall runs from 0.667 to 1.0 across the folds.
That spread is part of the answer, since each fold's recall rests on a dozen stations.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/09-imbalanced-classes-solutions.ipynb).

**1.** Fit a `DummyClassifier(strategy="stratified", random_state=0)`, which guesses frost at random in proportion to its share, and print its accuracy and recall on the test set.

In [10]:
# your code here

**2.** Print the unweighted model's precision and recall at a threshold of 0.2, using `predict_proba`, and compare them with the weighted model's at 0.5.

In [11]:
# your code here

**3.** Cross-validate the unweighted model with `StratifiedKFold(5, shuffle=True, random_state=0)` and `scoring="balanced_accuracy"`, and print the mean.

In [12]:
# your code here

**4.** Make a table with one row for each of the three models, the baseline, the model and the weighted model, and columns for accuracy, balanced accuracy, recall and precision on the test set, with `zero_division=0` for the baseline's precision.

In [13]:
# your code here

## Common errors

### UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples.

In [14]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    baseline_precision = precision_score(y_test, baseline_says)

print("precision:", baseline_precision)
for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).splitlines()[0]}")

precision: 0.0


Precision divides by the number of frost predictions, and the baseline made none, so there is nothing
to divide by. scikit-learn reports 0.0 and warns. The cell catches the warning to print it on one line.
`precision_score(..., zero_division=0)` sets the value and silences the warning, and seeing
the warning at all says the model never predicted the class.

### No error, and a test set with 10 frost stations or 21

In [15]:
for seed in range(6):
    _, _, _, unstratified_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    print("seed", seed, "->", int(unstratified_test.sum()), "frost stations in the test set")

seed 0 -> 21 frost stations in the test set
seed 1 -> 12 frost stations in the test set
seed 2 -> 14 frost stations in the test set
seed 3 -> 18 frost stations in the test set
seed 4 -> 10 frost stations in the test set
seed 5 -> 19 frost stations in the test set


Without `stratify`, each seed shuffled the stations differently, and the test set held anywhere from 10
to 21 of the 59. A recall on 10 stations moves in steps of a tenth. Pass `stratify=y` to every split of
an imbalanced class, as the worked example did.

### No error, and a recall of 0.0 or 1.0 in every fold

In [16]:
first_sixty = network.head(60)
print(int(first_sixty["june_frost"].sum()), "frost stations among the first 60")

print(cross_val_score(model, first_sixty[FEATURES], first_sixty["june_frost"], cv=StratifiedKFold(5),
                      scoring="recall"))

5 frost stations among the first 60
[1. 0. 1. 0. 0.]


Sixty stations hold five frosts, so each of the five folds tests one. A model either finds that one
or misses it, and the fold's recall is 1.0 or 0.0. The scores are honest, and they say almost nothing.
Gather more of the rare class before trusting any score on it, or use fewer folds so that each holds
more of it.

### No error, and a better model with a lower accuracy

In [17]:
print("model:   ", round(accuracy_score(y_test, says), 3), "accuracy,", round(recall_score(y_test, says), 3), "recall")
print("weighted:", round(accuracy_score(y_test, weighted_says), 3), "accuracy,", round(recall_score(y_test, weighted_says), 3), "recall")

model:    0.973 accuracy, 0.533 recall
weighted: 0.897 accuracy, 0.867 recall


Ranked by accuracy, the unweighted model wins and the weighted one loses, although the weighted model
finds 13 frosts where the other finds 8. `GridSearchCV` scores a classifier by accuracy unless told
otherwise, so a search over `class_weight` meets the same trap, and the **Hyperparameter Search**
notebook runs that search both ways.

## Recap

- Compare any model of a rare class with `DummyClassifier`, which predicts the common class everywhere
- Report recall, precision, balanced accuracy and average precision, which show the misses that
  accuracy hides
- Use `class_weight="balanced"` to trade accuracy for finding more of the rare class
- Keep the rare class's share in every split with `stratify` and `StratifiedKFold`
- Read a precision warning as a model that never predicted the class

## What is next

The **Hyperparameter Search** notebook tries many settings of a model at once, chooses among them by a
score, and measures how much that choice flatters the winner.

---

&#8592; **Previous:** [Classification Metrics](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/08-classification-metrics.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Hyperparameter Search](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/10-hyperparameter-search.ipynb) &#8594;
